In [1]:
#Jupiter Notebook'ta çalışır

import os

os.makedirs("/home/jovyan/data/bronze", exist_ok=True)
os.makedirs("/home/jovyan/data/checkpoints/bronze", exist_ok=True)

print("Çalışma dizini:", os.getcwd())
print("data içeriği:", os.listdir("/home/jovyan/data"))

Çalışma dizini: c:\Users\tahab\Desktop\BuyukVeri\notebooks
data içeriği: ['bronze', 'checkpoints']


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BronzeTest") \
    .master("local[*]") \
    .config("spark.jars.packages","org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1,io.delta:delta-spark_2.12:3.1.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

print("Spark:", spark.version)

Spark: 3.5.5


In [3]:
from pyspark.sql.types import *
from pyspark.sql.functions import from_json, col

schema = StructType([
    StructField("timestamp", StringType()),
    StructField("user_id", StringType()),
    StructField("event_type", StringType()),
    StructField("related_id", StringType()),
    StructField("data", StructType([
        StructField("iso_code", StringType()),
        StructField("country", StringType()),
        StructField("year", DoubleType()),
        StructField("primary_energy_consumption", DoubleType()),
        StructField("fossil_fuel_consumption", DoubleType()),
        StructField("renewables_consumption", DoubleType()),
        StructField("carbon_intensity_elec", DoubleType()),
    ]))
])

#Test için bu değer kullanıldı değiştirilebilir

df_raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "world_energy_consumption") \
    .option("startingOffsets", "earliest") \
    .load()

df_parsed = df_raw.select(
    from_json(col("value").cast("string"), schema).alias("msg")
).select("msg.*")

print("Stream hazır")

Stream hazır


In [ ]:
bronze_query = df_parsed.writeStream \
    .format("delta") \
    .option("checkpointLocation", "/data/checkpoints/bronze") \
    .outputMode("append") \
    .start("/data/bronze")

print("Veri toplanıyor")
bronze_query.awaitTermination(30)
print("Veri toplandı")

Veri toplanıyor


StreamingQueryException: [STREAM_FAILED] Query [id = 13cad422-b486-479c-83d5-354ef74f6963, runId = 787b1608-441d-4ba5-a34e-46ab4127d43e] terminated with exception: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)'

In [ ]:
df_bronze = spark.read.format("delta").load("/home/jovyan/data/bronze")
print("Toplam kayıt:", df_bronze.count())
df_bronze.show(5)

Toplam kayıt: 17432
+--------------------+-------+---------------+----------+--------------------+
|           timestamp|user_id|     event_type|related_id|                data|
+--------------------+-------+---------------+----------+--------------------+
|2026-05-10T16:05:...|  admin|Enerji Tüketimi|  EST-1999|{EST, Estonia, 19...|
|2026-05-10T16:05:...|  admin|Enerji Tüketimi|  AUS-1998|{AUS, Australia, ...|
|2026-05-10T16:05:...|  admin|Enerji Tüketimi|  JAM-2013|{JAM, Jamaica, 20...|
|2026-05-10T16:05:...|  admin|Enerji Tüketimi|  KAZ-1991|{KAZ, Kazakhstan,...|
|2026-05-10T16:05:...|  admin|Enerji Tüketimi|  NCL-1991|{NCL, New Caledon...|
+--------------------+-------+---------------+----------+--------------------+
only showing top 5 rows

